In [119]:
%config InlineBackend.figure_format = 'svg'

# Qiskit References

- Qiskit Guides: https://quantum.cloud.ibm.com/docs/en/guides
- Qiskit API Docs: https://quantum.cloud.ibm.com/docs/en/api/qiskit
- IBM Quantum Platform Classic: https://quantum.ibm.com/
- IBM Quantum Platform Classic Setup: https://docs.quantum.ibm.com/guides/setup-channel

In [120]:
import numpy as np
from numpy import pi
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import networkx as nx
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister, Parameter
from qiskit.quantum_info import Operator, Pauli, SparsePauliOp, Statevector, DensityMatrix
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import timeline_drawer

Below are an ideal sampler and estimator, for use in noiseless simulations.

In [121]:
from qiskit_aer.primitives import SamplerV2 as AerSampler, EstimatorV2 as AerEstimator
ideal_sampler = AerSampler()
ideal_estimator = AerEstimator()

To access real QPUs, such as IBM Kyiv below, you must set up an account on [IBM Quantum Platform Classic](https://docs.quantum.ibm.com/guides/setup-channel) or [IBM Cloud Account](https://quantum.cloud.ibm.com/docs/en/guides/cloud-setup).

In [122]:
# from qiskit_ibm_runtime import QiskitRuntimeService
# service = QiskitRuntimeService()
# ibm_kyiv = service.backend("ibm_kyiv")

Below are a sampler and estimator for noisy simulation on IBM Kyiv, as well as a pass manager for that backend.

In [123]:
# ibm_kyiv_aer_sampler = AerSampler.from_backend(ibm_kyiv)
# ibm_kyiv_aer_estimator = AerEstimator.from_backend(ibm_kyiv)
# ibm_kyiv_pm = generate_preset_pass_manager(optimization_level=3, backend=ibm_kyiv)

Below are a sampler and estimator for runtime execution on IBM Kyiv. 

In [124]:
# from qiskit_ibm_runtime import SamplerV2 as RuntimeSampler, EstimatorV2 as RuntimeEstimator
# ibm_kyiv_runtime_sampler = AerSampler.from_backend(ibm_kyiv)
# ibm_kyiv_runtime_estimator = AerEstimator.from_backend(ibm_kyiv)

Some utility functions:

In [125]:
from collections.abc import Sequence
from itertools import product

def bitstrings(num_bits: int) -> Sequence[str]:
    """The sequence of all bitstrings for the given number of bits."""
    return tuple(map("".join, product("01", repeat=num_bits)))

def paulistrings(num_qubits: int) -> Sequence[str]:
    """The sequence of all Paulistrings for the given number of qubits."""
    return tuple(map("".join, product("IXYZ", repeat=num_qubits)))

In [126]:
from collections.abc import Sequence 

def _balanced_cx_tree(qubits):
    cs, ts = [], []
    if len(qubits) <= 1:
        return cs, ts
    if len(qubits) % 2 == 1:
        cs.append(qubits[-1]); ts.append(qubits[-2])
        qubits = qubits[:-1]
    cs.extend(qubits[::2]); ts.extend(qubits[1::2])
    _cs, _ts = _balanced_cx_tree(qubits[1::2])
    cs.extend(_cs); ts.extend(_ts)
    return cs, ts

def apply_phase_gadget(circ: QuantumCircuit, legs: Sequence[int], angle: float | Parameter) -> None:
    if not legs:
        return
    if len(legs) == 1:
        circ.rz(angle, legs[0])
        return
    cs, ts = _balanced_cx_tree(legs)
    circ.cx(cs[:-1], ts[:-1])
    circ.rzz(angle, cs[-1], ts[-1])
    circ.cx(cs[-2::-1], ts[-2::-1])

def apply_pauli_gadget(circ: QuantumCircuit, pauli: Pauli, angle: float | Parameter) -> None:
    _paulis = str(pauli)[::-1]
    for q, p in enumerate(_paulis):
        if p == "X":
            circ.h(q)
        elif p == "Y":
            circ.sx(q)
    apply_phase_gadget(circ, [q for q, p in enumerate(_paulis) if p != "I"], angle)
    for q, p in enumerate(_paulis):
        if p == "X":
            circ.h(q)
        elif p == "Y":
            circ.sxdg(q)

In [127]:
from collections.abc import Iterator
from itertools import chain, combinations

def iter_powerset[T](s: Sequence[T]) -> Iterator[tuple[T, ...]]:
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def cost_to_problem_hamiltonian(cost: dict[str, float]) -> SparsePauliOp:
    assert cost
    num_qubits = len(next(iter(cost.keys())))
    assert all(len(bs) == num_qubits for bs in cost.keys())
    assert all(all(b in "01" for b in bs) for bs in cost.keys())
    return SparsePauliOp.from_sparse_list([
        ("Z"*len(legs), legs, coeff)
        for legs in iter_powerset(range(num_qubits))
        if (coeff := sum(c*(-1)**sum(int(bs[num_qubits-1-q]) for q in legs) for bs, c in cost.items())/(2**num_qubits)) != 0
    ], num_qubits=num_qubits)

# Assignment

## Task 1

In [128]:
from typing import Callable
from inspect import signature

In [129]:
def cost_func_to_dense_problem_hamiltonian(cost_function: Callable) -> SparsePauliOp:
    arity = len(signature(cost_function).parameters)
    return cost_to_problem_hamiltonian({
        ''.join(map(str, bit_tuple[::-1])): cost_function(*bit_tuple)
        for bit_tuple in product([0, 1], repeat=arity)
    })

def dense_to_sparse(problem_hamiltonian: SparsePauliOp, indices: list[int], num_qubits: int) -> SparsePauliOp:
    assert max(indices) < num_qubits, "Can't specify an index larger than the no. of qubits we have"
    return SparsePauliOp.from_sparse_list([
        (pauli, indices, coeff)
        for pauli, _, coeff in problem_hamiltonian.to_sparse_list()
    ], num_qubits)

def cost_func_to_sparse_problem_hamiltonian(cost_function: Callable, indices: list[int], num_qubits: int) -> SparsePauliOp:
    dense_ph = cost_func_to_dense_problem_hamiltonian(cost_function)
    return dense_to_sparse(dense_ph, indices, num_qubits)

In [130]:
def and_hamiltonian(i, j, k, num_qubits) -> SparsePauliOp:
    return cost_func_to_sparse_problem_hamiltonian(
        lambda bi, bj, bk: (bi and bj) ^ bk, [i, j, k], num_qubits
    )

def or_hamiltonian(i, j, k, num_qubits) -> SparsePauliOp:
    return cost_func_to_sparse_problem_hamiltonian(
        lambda bi, bj, bk: (bi or bj) ^ bk, [i, j, k], num_qubits
    )

def xor_hamiltonian(i, j, k, num_qubits) -> SparsePauliOp:
    return cost_func_to_sparse_problem_hamiltonian(
        lambda bi, bj, bk: (bi ^ bj) ^ bk, [i, j, k], num_qubits
    )

def not_hamiltonian(i, j, num_qubits) -> SparsePauliOp:
    return cost_func_to_sparse_problem_hamiltonian(
        lambda bi, bj: (not bi) ^ bj, [i, j], num_qubits
    )

def const0_hamiltonian(i, num_qubits) -> SparsePauliOp:
    return cost_func_to_sparse_problem_hamiltonian(
        lambda bi: 0 ^ bi, [i], num_qubits
    )

def const1_hamiltonian(i, num_qubits) -> SparsePauliOp:
    return cost_func_to_sparse_problem_hamiltonian(
        lambda bi: 1 ^ bi, [i], num_qubits
    )

In [131]:
xor_hamiltonian(0,2,7, num_qubits=8)

SparsePauliOp(['IIIIIIII', 'ZIIIIZIZ'],
              coeffs=[ 0.5+0.j, -0.5+0.j])

In [132]:
const1_hamiltonian(2, num_qubits=8)

SparsePauliOp(['IIIIIIII', 'IIIIIZII'],
              coeffs=[0.5+0.j, 0.5+0.j])